# Tigerfish Self-Play Veri Uretimi (Colab)

Colab'in CPU kaynaklarini kullanarak binpack egitim verisi uretir.
Tigerfish engine'i Colab'da derleyip self-play calistirir.

**Not:** Self-play CPU-yogun bir islemdir, GPU gerektirmez.
Colab Pro ile daha fazla CPU suresi elde edersiniz.

**Tavsiye:** Buyuk veri uretimi icin yerel makinenizi kullanin.
Bu notebook kucuk miktarlarda ek veri icin faydalidir.

## 1. Tigerfish Kaynak Kodunu Yukle

In [ ]:
#@title Kaynak Kod Yukle { run: "auto" }
SOURCE_METHOD = "drive"  #@param ["github", "drive", "upload"]

import os

if SOURCE_METHOD == "github":
    REPO_URL = ""  #@param {type:"string"}
    if REPO_URL:
        !git clone --depth 1 "$REPO_URL" /content/tigerfish
    else:
        print("HATA: GitHub repo URL'si giriniz.")

elif SOURCE_METHOD == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_SRC = "/content/drive/MyDrive/tigerfish/src"  #@param {type:"string"}
    if os.path.isdir(DRIVE_SRC):
        !mkdir -p /content/tigerfish
        !cp -r "$DRIVE_SRC" /content/tigerfish/src
        print(f"Kaynak kod kopyalandi: {DRIVE_SRC}")
    else:
        print(f"HATA: {DRIVE_SRC} bulunamadi!")

elif SOURCE_METHOD == "upload":
    print("Tigerfish src/ klasorunu zip olarak yukleyin:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.zip'):
            !mkdir -p /content/tigerfish && unzip -o "$fname" -d /content/tigerfish
            print(f"Zip acildi: {fname}")
        elif fname.endswith('.tar.gz'):
            !mkdir -p /content/tigerfish && tar xzf "$fname" -C /content/tigerfish
            print(f"Tarball acildi: {fname}")

## 2. Tigerfish Derle

In [ ]:
%%bash
cd /content/tigerfish/src

# NNUE net dosyalarinin varligini kontrol et
ls nn-*.nnue 2>/dev/null || echo "UYARI: .nnue dosyalari bulunamadi, derleme basarisiz olabilir!"

# Derle
make -j$(nproc) ARCH=x86-64-avx2 COMP=gcc all 2>&1 | tail -5

# Test
echo "" | timeout 5 ./tigerfish 2>/dev/null | head -1 || echo "Engine calismadi!"

## 3. Self-Play Veri Uretimi

In [ ]:
#@title Self-Play Parametreleri { run: "auto" }
DEPTH = 9         #@param {type:"integer"}
COUNT = 1000000   #@param {type:"integer"}
THREADS = 2       #@param {type:"integer"}
HASH_MB = 128     #@param {type:"integer"}
RANDOM_MOVES = 8  #@param {type:"integer"}
EVAL_LIMIT = 3000 #@param {type:"integer"}

# Colab CPU sayisini goster
import multiprocessing
print(f"Colab CPU: {multiprocessing.cpu_count()} core")
print(f"Kullanilacak thread: {THREADS}")
print(f"Hedef: {COUNT:,} pozisyon, depth {DEPTH}")

In [ ]:
import subprocess, time, os

DATA_DIR = "/content/training_data"
os.makedirs(DATA_DIR, exist_ok=True)
OUTPUT = os.path.join(DATA_DIR, "tiger_colab_selfplay")

uci_commands = f"""uci
setoption name Threads value {THREADS}
setoption name Hash value {HASH_MB}
isready
generate_training_data depth {DEPTH} count {COUNT} random_move_count {RANDOM_MOVES} eval_limit {EVAL_LIMIT} output_file_name {OUTPUT}
quit
"""

print("=" * 60)
print(f"  Self-Play Basliyor: {COUNT:,} pozisyon, depth {DEPTH}")
print("=" * 60)

start = time.time()
proc = subprocess.Popen(
    ["/content/tigerfish/src/tigerfish"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
proc.stdin.write(uci_commands)
proc.stdin.flush()
proc.stdin.close()

for line in proc.stdout:
    print(line, end="")

proc.wait()
elapsed = time.time() - start

binpack = OUTPUT + ".binpack"
if os.path.isfile(binpack):
    size_mb = os.path.getsize(binpack) / 1024**2
    print(f"\nTamamlandi: {size_mb:.1f} MB, {elapsed/60:.1f} dakika")
else:
    print(f"\nHATA: Cikti dosyasi bulunamadi!")

## 4. Veriyi Kaydet

In [ ]:
#@title Kaydetme Yontemi { run: "auto" }
SAVE_METHOD = "drive"  #@param ["download", "drive"]

binpack = OUTPUT + ".binpack"

if not os.path.isfile(binpack):
    print("Binpack dosyasi bulunamadi!")
elif SAVE_METHOD == "drive":
    import shutil
    DRIVE_DATA = "/content/drive/MyDrive/tigerfish/training_data"  #@param {type:"string"}
    os.makedirs(DRIVE_DATA, exist_ok=True)
    dest = os.path.join(DRIVE_DATA, os.path.basename(binpack))
    shutil.copy2(binpack, dest)
    print(f"Drive'a kaydedildi: {dest} ({os.path.getsize(dest)/1024**2:.1f} MB)")
elif SAVE_METHOD == "download":
    from google.colab import files
    files.download(binpack)

---

## Sonraki Adim

Uretilen `.binpack` dosyasini egitim notebook'una (`01_tiger_train.ipynb`) yukleyerek
NNUE fine-tuning islemini baslatin.